In [1]:
%env CUDA_VISIBLE_DEVICES=0,1,2

env: CUDA_VISIBLE_DEVICES=0,1,2


In [4]:
from pathlib import Path
import json
import sys
import torch

from IPython.display import Video, display
import sys
sys.path.append("/home/gaoya/Code_Video/DiffSynth-Studio-main")
from diffsynth.pipelines.wan_video import WanVideoPipeline, ModelConfig

TRAIN0419_ROOT = Path("/home/gaoya/Code_Video/Code_data/Code_train/train_0419")

from batch_eval_lora import (
    MODEL_ROOT,
    build_pipeline,
    collect_cases,
    generate_one_video,
    align_generation_num_frames,
    save_video,
)


/data/gaoya/miniconda3/envs/wan/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
I0000 00:00:1777468438.812508 2290007 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777468438.876606 2290007 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777468440.308112 2290007 port.cc:153] oneDNN custom operations are on. You may see slightly different n

In [5]:
# 只需要改这一组配置
WAN_ROOT = MODEL_ROOT
VACE_ROOT = Path("/home/gaoya/ckpt_cache/Wan-AI-Wan2.1-VACE-1.3B")
LORA_PATH = Path("/data/gaoya/AAA_test_video/Train_test/DiffSynth_wan22_ti2v5B/openvid_mixed_ctx24_384x672_lora/checkpoints/step-008000/checkpoint.safetensors")


DEVICE0 = "cuda:0"
DEVICE1 = "cuda:1"
VACE_DEVICE = DEVICE1
HEIGHT = 384
WIDTH = 672
NUM_FRAMES = 72
CONTEXT_FRAMES = 8
FPS = 16
NUM_INFERENCE_STEPS = 50
CFG_SCALE = 5.0
SEED = 4
QUALITY = 5



NEGATIVE_PROMPT = ""




OUTPUT_VIDEO_PATH = Path(
    "/data/gaoya/AAA_test_video/Train_test/DiffSynth_wan22_ti2v5B/openvid_mixed_ctx24_384x672_lora/test_tmp/lora.mp4"
    )
OUTPUT_VIDEO_PATH_ori = Path(
        "/data/gaoya/AAA_test_video/Train_test/DiffSynth_wan22_ti2v5B/openvid_mixed_ctx24_384x672_lora/test_tmp/ori.mp4"
)
VACE_OUTPUT_VIDEO_PATH = Path(
    "/data/gaoya/AAA_test_video/Train_test/DiffSynth_wan22_ti2v5B/openvid_mixed_ctx24_384x672_lora/test_tmp/vace.mp4"
)
OUTPUT_VIDEO_PATH.parent.mkdir(parents=True, exist_ok=True)

assert WAN_ROOT.exists(), WAN_ROOT
assert VACE_ROOT.exists(), VACE_ROOT
assert LORA_PATH.exists(), LORA_PATH


aligned_num_frames = align_generation_num_frames(NUM_FRAMES)
print({
    "wan_root": str(WAN_ROOT),
    "lora_path": str(LORA_PATH),
    "vace_root": str(VACE_ROOT),
    "output_video_path": str(OUTPUT_VIDEO_PATH),
    "device": DEVICE1,
    "vace_device": VACE_DEVICE,
    "size": [HEIGHT, WIDTH],
    "requested_num_frames": NUM_FRAMES,
    "aligned_generation_num_frames": aligned_num_frames,
    "context_frames": CONTEXT_FRAMES,
    "fps": FPS,
    "num_inference_steps": NUM_INFERENCE_STEPS,
    "cfg_scale": CFG_SCALE,
    "seed": SEED,
})


{'wan_root': '/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B', 'lora_path': '/data/gaoya/AAA_test_video/Train_test/DiffSynth_wan22_ti2v5B/openvid_mixed_ctx24_384x672_lora/checkpoints/step-008000/checkpoint.safetensors', 'vace_root': '/home/gaoya/ckpt_cache/Wan-AI-Wan2.1-VACE-1.3B', 'output_video_path': '/data/gaoya/AAA_test_video/Train_test/DiffSynth_wan22_ti2v5B/openvid_mixed_ctx24_384x672_lora/test_tmp/lora.mp4', 'device': 'cuda:1', 'vace_device': 'cuda:1', 'size': [384, 672], 'requested_num_frames': 72, 'aligned_generation_num_frames': 73, 'context_frames': 8, 'fps': 16, 'num_inference_steps': 50, 'cfg_scale': 5.0, 'seed': 4}


In [6]:
pipe0 = build_pipeline(
    wan_root=WAN_ROOT,
    device=DEVICE0,
    lora_path=LORA_PATH,
)
pipe_ori = build_pipeline(
    wan_root=WAN_ROOT,
    device=DEVICE1,
    lora_path = None,
)
print("Pipeline ready.")
# 可选: 单独加载 Wan2.1-VACE-1.3B
# 默认关闭，避免和上面的两个 pipeline 同时占满显存。
# 需要时把 LOAD_VACE 改成 True，再运行这个单元。

LOAD_VACE = False
pipe_vace = None

if LOAD_VACE:
    pipe_vace = WanVideoPipeline.from_pretrained(
        torch_dtype=torch.bfloat16,
        device=VACE_DEVICE,
        model_configs=[
            ModelConfig(path=str(VACE_ROOT / "diffusion_pytorch_model.safetensors")),
            ModelConfig(path=str(VACE_ROOT / "models_t5_umt5-xxl-enc-bf16.pth")),
            ModelConfig(path=str(VACE_ROOT / "Wan2.1_VAE.pth")),
        ],
        tokenizer_config=ModelConfig(
            path=str(VACE_ROOT / "google" / "umt5-xxl"),
            skip_download=True,
        ),
        redirect_common_files=False,
    )
    print("VACE pipeline ready.")
else:
    print("Set LOAD_VACE=True to load Wan2.1-VACE-1.3B.")


Loading models from: [
    "/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B/diffusion_pytorch_model-00001-of-00003.safetensors",
    "/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B/diffusion_pytorch_model-00002-of-00003.safetensors",
    "/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B/diffusion_pytorch_model-00003-of-00003.safetensors"
]
Loaded model: {
    "model_name": "wan_video_dit",
    "model_class": "diffsynth.models.wan_video_dit.WanModel",
    "extra_kwargs": {
        "has_image_input": false,
        "patch_size": [
            1,
            2,
            2
        ],
        "in_dim": 48,
        "dim": 3072,
        "ffn_dim": 14336,
        "freq_dim": 256,
        "text_dim": 4096,
        "out_dim": 48,
        "num_heads": 24,
        "num_layers": 30,
        "eps": 1e-06,
        "seperated_timestep": true,
        "require_clip_embedding": false,
        "require_vae_embedding": false,
        "fuse_vae_embedding_in_latents": true
    }
}
Loading models from: "/data/gaoya/ckpt/Wan-AI-Wan2

In [7]:

META_JSON_PATH = Path(
    # "/data/gaoya/AAA_test_video/Dataset_physV/0417data/version_1_genesis_rigid_data_all_cases/mytest/genesis_heldout_0008__10005__case007_entry_fast_center/meta.json"
    "/data/gaoya/dataset/physics-iq-benchmark/mytest/0002_perspective-center_trimmed-ball-and-block-fall/meta.json"
    )

In [8]:
case = collect_cases([META_JSON_PATH], limit=1)[0]
print(json.dumps({
    "dataset": case["dataset"],
    "sample_id": case["sample_id"],
    "context_path": case["context_path"],
    "future_gt_path": case.get("future_gt_path"),
    "full_video_path": case.get("full_video_path"),
    "context_resize_mode": case.get("context_resize_mode"),
    "prompt": case["caption"],
}, ensure_ascii=False, indent=2))


{
  "dataset": "physics-iq-benchmark",
  "sample_id": "0002_perspective-center_trimmed-ball-and-block-fall",
  "context_path": "/data/gaoya/dataset/physics-iq-benchmark/mytest/0002_perspective-center_trimmed-ball-and-block-fall/context_video.mp4",
  "future_gt_path": "/data/gaoya/dataset/physics-iq-benchmark/mytest/0002_perspective-center_trimmed-ball-and-block-fall/future_gt_video.mp4",
  "full_video_path": "/data/gaoya/dataset/physics-iq-benchmark/full-videos/take-1/30FPS/0002_full-videos_30FPS_perspective-center_take-1_trimmed-ball-and-block-fall.mp4",
  "context_resize_mode": "crop",
  "prompt": "Two pillows on a table and two grabber tools hanging above them from which a brown tennis ball and an orange block are suspended. The grabber tools let go of the ball and block. Static shot with no camera movement."
}


In [ ]:
video, used_context_frames = generate_one_video(
    pipe=pipe0,
    context_path=Path(case["context_path"]),
    prompt=case["caption"],
    negative_prompt=NEGATIVE_PROMPT,
    seed=SEED,
    height=HEIGHT,
    width=WIDTH,
    num_frames=aligned_num_frames,
    fps=FPS,
    cfg_scale=CFG_SCALE,
    num_inference_steps=NUM_INFERENCE_STEPS,
    context_frames=CONTEXT_FRAMES,
    output_num_frames=NUM_FRAMES,
    context_resize_mode=case.get("context_resize_mode", "crop"),
)
save_video(video, str(OUTPUT_VIDEO_PATH), fps=FPS, quality=QUALITY)


VAE decoding:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:

video_ori, used_context_frames_ori = generate_one_video(
    pipe=pipe_ori,
    context_path=Path(case["context_path"]),
    prompt=case["caption"],
    negative_prompt=NEGATIVE_PROMPT,
    seed=SEED,
    height=HEIGHT,
    width=WIDTH,
    num_frames=aligned_num_frames,
    fps=FPS,
    cfg_scale=CFG_SCALE,
    num_inference_steps=NUM_INFERENCE_STEPS,
    context_frames=CONTEXT_FRAMES,
    output_num_frames=NUM_FRAMES,
    context_resize_mode=case.get("context_resize_mode", "crop"),
)

save_video(video_ori, str(OUTPUT_VIDEO_PATH_ori), fps=FPS, quality=QUALITY)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Saving video:   0%|          | 0/72 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Saving video: 100%|██████████| 72/72 [00:00<00:00, 453.73it/s]


In [15]:
print(OUTPUT_VIDEO_PATH)
print(OUTPUT_VIDEO_PATH_ori)
print(VACE_OUTPUT_VIDEO_PATH)


/data/gaoya/AAA_test_video/Train_test/DiffSynth_wan22_ti2v5B/openvid_mixed_ctx24_384x672_lora/test_tmp/lora.mp4
/data/gaoya/AAA_test_video/Train_test/DiffSynth_wan22_ti2v5B/openvid_mixed_ctx24_384x672_lora/test_tmp/ori.mp4
/data/gaoya/AAA_test_video/Train_test/DiffSynth_wan22_ti2v5B/openvid_mixed_ctx24_384x672_lora/test_tmp/vace.mp4


In [14]:
case["caption"]

'Two pillows on a table and two grabber tools hanging above them from which a brown tennis ball and an orange block are suspended. The grabber tools let go of the ball and block. Static shot with no camera movement.'

In [13]:
print(case["context_path"])

/data/gaoya/dataset/physics-iq-benchmark/mytest/0002_perspective-center_trimmed-ball-and-block-fall/context_video.mp4
